# 3. Dimensionality reduction: motif alignment <a id="3"></a>
In this section we will apply dimensionality reduction to our coarse-grained representation of selected activation loops.


## Table of contents

- [3.1 Activation loop alignment](#32)


## Backend map

How this notebook connects to `workflow/` modules:

```mermaid
flowchart LR
  nb["04a-MotifAlignment"]
  m0["workflow.align"]
  nb --> m0
  m1["workflow.ca_stripper"]
  nb --> m1
  m2["workflow.utilities"]
  nb --> m2
```


![State of the workflow](images/DimensionalityReduction.png)

To get started, let's load some packages!

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from workflow.align import Alignment
from workflow.ca_stripper import OutlierStripper
from workflow.utilities import PDBDownloader
from workflow.utilities import (
    count_pdb_files,
    braf_res,
    clear_and_make,
    make_seg,
    copy_filtered_pdbs,
    copy_cg_chain_small_molecules,
)


## 3.1 Activation loop alignment  <a id="32"></a>
The aim of this step is to find a structurally-viable alignment of the loop extremities in order to minimise the impact of the lack of roto-translational invariance on PCA performance.

We perform a least-squares structural superposition of each kinase in our dataset with the reference BRAF structure, by minimising the root-mean-square deviation of their Cα atoms.

In [ ]:
# Import the class
from workflow.align import Alignment

# Initialise the aligner
aligner = Alignment()

# Reference structure (same directory as this notebook)
reference_pdb = "6UAN_chainD.pdb"

In [ ]:
# Motif-based alignment (DFG+APE CA atoms)
# NOTE: the method name is kept for notebook compatibility.
aligner.process_pymol_alignment(
    pdb_dir="Results/Bounds_CAfilter_chains/",
    reference_pdb=reference_pdb,
    output_dir="Results/activation_segments/aligned_mda/",
    ref_name="6UAN_chainD",
    quiet=True,  # tqdm + RMSD only; suppress save/reference chatter and warnings
)

We now retain loops whose DFG and APE extremities are aligned to the reference BRAF structure. For each aligned full chain, the activation-loop segment is extracted temporarily and its first and last Cα positions are compared with the reference. Structures whose maximum terminal distance exceeds 10 Å are excluded. This is a distance-only quality filter.

In [ ]:
from workflow.ca_stripper import OutlierStripper

extremity_filter = OutlierStripper(
    reference_pdb="6UAN_chainD.pdb",
    ref_first_resid=144,
    ref_last_resid=173,
)

extremity_results = extremity_filter.filter_chains_by_loop_termini(
    input_chains_dir="Results/activation_segments/aligned_mda/",
    output_chains_dir="Results/activation_segments/misaligned_filter/",
    motifs=["DFG", "APE"],
    distance_cutoff=10.0,
    create_plots=True,
)


Let's check how many kinase loops are in the newly created directories. 

In [ ]:
pdb_directory = "Results/activation_segments/aligned_mda/"
pdb_directory2 = "Results/activation_segments/misaligned_filter/"
pdb_count = count_pdb_files(pdb_directory)
pdb_count2 = count_pdb_files(pdb_directory2)

print(f"There are {pdb_count} PDB files in the directory '{pdb_directory}'.")
print(f"There are {pdb_count2} PDB files in the directories '{pdb_directory2}'.")